In [ ]:
# Structure features with top-2.5% structure perturbation

from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments

def encode_structure_symbol(symbol):
    if symbol == '.':
        return [1, 0, 0]
    elif symbol == '(':
        return [0, 1, 0]
    elif symbol == ')':
        return [0, 0, 1]
    else:
        return [0, 0, 0]

def get_structure_string(subopt_result, structure_rank=0):
    """
    structure_rank = 0: highest-probability / lowest-energy structure
    structure_rank = 1: second-highest-probability structure
    """
    if len(subopt_result) > structure_rank:
        return str(subopt_result[structure_rank].structure)
    else:
        return str(subopt_result[0].structure)

def calculate_probability_gap(subopt_result, strands, model):
    if len(subopt_result) < 2:
        return 0.0

    prob_top = structure_probability(
        strands=strands,
        structure=subopt_result[0].structure,
        model=model
    )

    prob_second = structure_probability(
        strands=strands,
        structure=subopt_result[1].structure,
        model=model
    )

    return prob_top - prob_second

import re

file_paths = [
    'Feature_CNN2_0025.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('HT1-1_full_guide_sequences.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('HT1-1_target_sequences_noPAM.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('HT1-1_target_sequences_noPAM.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

# ============================================================
# First pass:
# calculate top-vs-second structure probability gap
# ============================================================

guide_gap_list = []
target_gap_list = []

all_guide_bh_subopts = []
all_target_bh_subopts = []
all_hybrid_ah_subopts = []
all_target_ah_subopts = []

for i in range (0, len(guide_array)):

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Compute suboptimal structures and energy
    subopt_structures_guide_bh = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target_bh = subopt(strands=[target_truncated, DNA_reverse_complement(target_truncated)], energy_gap=3, model=my_model_DNA)
    subopt_structures_hybrid_ah = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_DNA)
    subopt_structures_target_ah = subopt(strands=DNA_reverse_complement(target_truncated), energy_gap=3, model=my_model_DNA)

    all_guide_bh_subopts.append(subopt_structures_guide_bh)
    all_target_bh_subopts.append(subopt_structures_target_bh)
    all_hybrid_ah_subopts.append(subopt_structures_hybrid_ah)
    all_target_ah_subopts.append(subopt_structures_target_ah)

    guide_gap = calculate_probability_gap(
        subopt_structures_guide_bh,
        strands=guide,
        model=my_model_RNA
    )
    
    target_gap = calculate_probability_gap(
        subopt_structures_target_ah,
        strands=DNA_reverse_complement(target_truncated),
        model=my_model_DNA
    )

    guide_gap_list.append((i, guide_gap))
    target_gap_list.append((i, target_gap))

# Rank by largest probability gap
guide_gap_list_sorted = sorted(
    guide_gap_list,
    key=lambda x: x[1],
    reverse=True
)

target_gap_list_sorted = sorted(
    target_gap_list,
    key=lambda x: x[1],
    reverse=True
)

top_percent_count = max(1, math.ceil(len(guide_array) * 0.025))

guide_switch_indices = set(
    idx for idx, gap in guide_gap_list_sorted[:top_percent_count]
)

target_switch_indices = set(
    idx for idx, gap in target_gap_list_sorted[:top_percent_count]
)

print("Top 1% guide_bh samples switched to second-highest-probability structure:")
print(sorted(guide_switch_indices))

print("Top 1% target_bh samples switched to second-highest-probability structure:")
print(sorted(target_switch_indices))

# ============================================================
# Second pass:
# encode structures
# ============================================================

for i in range (0, len(guide_array)):

    # Initialize struct_array
    struct_array = []
    struct_array_unit = []

    subopt_structures_guide_bh = all_guide_bh_subopts[i]
    subopt_structures_target_bh = all_target_bh_subopts[i]
    subopt_structures_hybrid_ah = all_hybrid_ah_subopts[i]
    subopt_structures_target_ah = all_target_ah_subopts[i]

    # Use second-highest-probability structure for top 1% largest-gap guide_bh samples
    if i in guide_switch_indices:
        guide_bh_structure = get_structure_string(
            subopt_structures_guide_bh,
            structure_rank=1
        )
    else:
        guide_bh_structure = get_structure_string(
            subopt_structures_guide_bh,
            structure_rank=0
        )

    target_bh_structure = get_structure_string(
        subopt_structures_target_bh,
        structure_rank=0
    )

    hybrid_ah_structure = get_structure_string(
        subopt_structures_hybrid_ah,
        structure_rank=0
    )

    # Use second-highest-probability structure for top 1% largest-gap target_bh samples
    if i in target_switch_indices:
        target_ah_structure = get_structure_string(
            subopt_structures_target_ah,
            structure_rank=1
        )
    else:
        target_ah_structure = get_structure_string(
            subopt_structures_target_ah,
            structure_rank=0
        )

    # Guide structure before hybridization
    for j in range(40):
        try:
            to_be_added = encode_structure_symbol(guide_bh_structure[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)

    # Target structure before hybridization
    target_bh_segments = get_segments_interrupted_by_plus(target_bh_structure)

    target_bh_1 = target_bh_segments[0]
    target_bh_2 = target_bh_segments[1]

    for j in range(20):
        try:
            to_be_added = encode_structure_symbol(target_bh_1[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)

    for j in range(20):
        try:
            to_be_added = encode_structure_symbol(target_bh_2[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)

    # Hybrid structure after hybridization
    hybrid_ah_segments = get_segments_interrupted_by_plus(hybrid_ah_structure)

    hybrid_ah_1 = hybrid_ah_segments[0]
    hybrid_ah_2 = hybrid_ah_segments[1]

    for j in range(40):
        try:
            to_be_added = encode_structure_symbol(hybrid_ah_1[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)

    for j in range(20):
        try:
            to_be_added = encode_structure_symbol(hybrid_ah_2[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)

    # Target structure after hybridization
    for j in range(20):
        try:
            to_be_added = encode_structure_symbol(target_ah_structure[j])
        except IndexError:
            to_be_added = [0, 0, 0]

        struct_array_unit.append(to_be_added)
     
    struct_array_to_append = [struct_array_unit]
    struct_array.append(struct_array_to_append)

    struct_array = list(itertools.chain.from_iterable(struct_array))
    
    # Open a file in write mode
    with open('Feature_CNN2_0025.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

In [ ]:
# Energy features
# Gaussian noise: Mean = 0, STD = 0.05 (after normalization)

from nupack import *
import RNA
import math
import itertools
import numpy as np

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))

def RNA_to_DNA(RNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'U': 'T'}
    return ''.join(match.get(base, base) for base in (RNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def normalize_guide(value):
    normalized_value = (value - (-15)) / (1 - (-15))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-37)) / (-19 - (-37))
    return normalized_value

def normalize_guide_conse(value):
    normalized_value = (value - (-32)) / (2 - (-32))
    return normalized_value

def normalize_target_conse(value):
    normalized_value = (value - (-36)) / (-18 - (-36))
    return normalized_value

def normalize_guide_conse_unpaired(value):
    normalized_value = (value - (-51)) / (3 - (-51))
    return normalized_value

def normalize_target_conse_unpaired(value):
    normalized_value = (value - (-15)) / (0 - (-15))
    return normalized_value

def normalize_guide_overhang(value):
    normalized_value = (value - (-51)) / (3 - (-51))
    return normalized_value

def normalize_target_overhang(value):
    normalized_value = (value - (-15)) / (0 - (-15))
    return normalized_value

def normalize_guide_paired(value):
    normalized_value = (value - (-30)) / (3 - (-30))
    return normalized_value

def normalize_target_paired(value):
    normalized_value = (value - (-36)) / (-18 - (-36))
    return normalized_value

def normalize_seed(value):
    normalized_value = (value - (-11)) / (-3 - (-11))
    return normalized_value

def normalize_middle(value):
    normalized_value = (value - (-13)) / (-4 - (-13))
    return normalized_value

def normalize_distal(value):
    normalized_value = (value - (-14)) / (-4 - (-14))
    return normalized_value

def find_max_base_pairs(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '()':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def find_max_unpaired(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '.':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def detect_5_overhang(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_overhang(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def detect_5_paired(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_paired(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def is_all_parens(s):
    for char in s:
        if char not in ("(", ")"):
            return False
    return True
    

file_paths = [
    'Feature_MLP_0025.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('HT1-1_full_guide_sequences.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('HT1-1_target_sequences_noPAM.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('HT1-1_target_sequences_noPAM.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf

min_guide_energy_conse = np.inf
max_guide_energy_conse = -np.inf
min_target_energy_conse = np.inf
max_target_energy_conse = -np.inf

min_guide_energy_conse_unpaired = np.inf
max_guide_energy_conse_unpaired = -np.inf
min_target_energy_conse_unpaired = np.inf
max_target_energy_conse_unpaired = -np.inf

min_guide_energy_5_overhang = np.inf
max_guide_energy_5_overhang = -np.inf
min_target_energy_5_overhang = np.inf
max_target_energy_5_overhang = -np.inf

min_guide_energy_3_overhang = np.inf
max_guide_energy_3_overhang = -np.inf
min_target_energy_3_overhang = np.inf
max_target_energy_3_overhang = -np.inf

min_guide_energy_5_paired = np.inf
max_guide_energy_5_paired = -np.inf
min_target_energy_5_paired = np.inf
max_target_energy_5_paired = -np.inf

min_guide_energy_3_paired = np.inf
max_guide_energy_3_paired = -np.inf
min_target_energy_3_paired = np.inf
max_target_energy_3_paired = -np.inf

max_seed_energy = -np.inf
min_seed_energy = np.inf

max_middle_energy = -np.inf
min_middle_energy = np.inf

max_distal_energy = -np.inf
min_distal_energy = np.inf

 # Initialize energy_array
energy_array = []

# ===== Gaussian-noise settings =====

NOISE_STD = 0.05      # 5% standard deviation
NOISE_PERCENT = 0.025  # perturb 2.5% of samples for each feature independently

num_samples = len(guide_array)
num_features = 17

# Random sample indices for each feature
feature_noise_indices = []

for feature_idx in range(num_features):
    indices = np.random.choice(
        num_samples,
        size=max(1, int(np.ceil(num_samples * NOISE_PERCENT))),
        replace=False
    )
    feature_noise_indices.append(set(indices))

for i in range (0, len(guide_array)):
    energy_array_unit = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Initialize energy_array
    energy_array_guide = []
    energy_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_DNA)
    ensemble_energy_target = partition_function_target[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=6, model=my_model_RNA)  
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=6, model=my_model_DNA)

    len_scaffold = 20
    len_spacer = 20
    
    # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy
    if subopt_structures_guide[0].energy + 3.57 > max_guide_energy:
        max_guide_energy = subopt_structures_guide[0].energy + 3.57
    if subopt_structures_guide[0].energy + 3.57 < min_guide_energy:
        min_guide_energy = subopt_structures_guide[0].energy + 3.57
    if subopt_structures_target[0].energy > max_target_energy:
        max_target_energy = subopt_structures_target[0].energy
    if subopt_structures_target[0].energy < min_target_energy:
        min_target_energy = subopt_structures_target[0].energy
        
    # Calculate/compare ensemble_energy_max_paired_guide and ensemble_energy_max_paired_target
    if str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_base_pairs(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_max_paired_guide = pfunc(strands=[max_paired_seq_guide, RNA_reverse_complement(max_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_paired_guide = partition_function_max_paired_guide[1]

    if str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_base_pairs(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_max_paired_seq_target = RNA_to_DNA(max_paired_seq_target)
        partition_function_max_paired_target = pfunc(strands=[DNA_max_paired_seq_target, DNA_reverse_complement(DNA_max_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_paired_target = partition_function_max_paired_target[1]

    if ensemble_energy_max_paired_guide > max_guide_energy_conse:
        max_guide_energy_conse = ensemble_energy_max_paired_guide
    if ensemble_energy_max_paired_guide < min_guide_energy_conse:
        min_guide_energy_conse = ensemble_energy_max_paired_guide

    if ensemble_energy_max_paired_target > max_target_energy_conse:
        max_target_energy_conse = ensemble_energy_max_paired_target
    if ensemble_energy_max_paired_target < min_target_energy_conse:
        min_target_energy_conse = ensemble_energy_max_paired_target

    
    # Calculate/compare ensemble_energy_max_unpaired_guide and ensemble_energy_max_unpaired_target
    if is_all_parens(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer]) == True:
        ensemble_energy_max_unpaired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_unpaired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_unpaired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_max_unpaired_guide = pfunc(strands=[max_unpaired_seq_guide, RNA_reverse_complement(max_unpaired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_unpaired_guide = partition_function_max_unpaired_guide[1]

    if is_all_parens(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer]) == True:
        ensemble_energy_max_unpaired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_unpaired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_unpaired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_max_unpaired_seq_target = RNA_to_DNA(max_unpaired_seq_target)
        partition_function_max_unpaired_target = pfunc(strands=[DNA_max_unpaired_seq_target, DNA_reverse_complement(DNA_max_unpaired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_unpaired_target = partition_function_max_unpaired_target[1]

    if ensemble_energy_max_unpaired_guide > max_guide_energy_conse_unpaired:
        max_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide
    if ensemble_energy_max_unpaired_guide < min_guide_energy_conse_unpaired:
        min_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide

    if ensemble_energy_max_unpaired_target > max_target_energy_conse_unpaired:
        max_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target
    if ensemble_energy_max_unpaired_target < min_target_energy_conse_unpaired:
        min_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target

    # Calculate/compare ensemble_energy_5_overhang_guide and ensemble_energy_5_overhang_target
    if str(subopt_structures_guide[0].structure)[len_scaffold] != '.':
        ensemble_energy_5_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_overhang(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_5_overhang_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_5_overhang_guide = pfunc(strands=[max_5_overhang_seq_guide, RNA_reverse_complement(max_5_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_5_overhang_guide = partition_function_5_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[len_scaffold] != '.':
        ensemble_energy_5_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_overhang(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_5_overhang_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_5_overhang_seq_target = RNA_to_DNA(max_5_overhang_seq_target)
        partition_function_5_overhang_target = pfunc(strands=[DNA_5_overhang_seq_target, DNA_reverse_complement(DNA_5_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_5_overhang_target = partition_function_5_overhang_target[1]

    if ensemble_energy_5_overhang_guide > max_guide_energy_5_overhang:
        max_guide_energy_5_overhang = ensemble_energy_5_overhang_guide
    if ensemble_energy_5_overhang_guide < min_guide_energy_5_overhang:
        min_guide_energy_5_overhang = ensemble_energy_5_overhang_guide

    if ensemble_energy_5_overhang_target > max_target_energy_5_overhang:
        max_target_energy_5_overhang = ensemble_energy_5_overhang_target
    if ensemble_energy_5_overhang_target < min_target_energy_5_overhang:
        min_target_energy_5_overhang = ensemble_energy_5_overhang_target
            

    # Calculate/compare ensemble_energy_3_overhang_guide and ensemble_energy_3_overhang_target
    if str(subopt_structures_guide[0].structure)[len_scaffold+len_spacer-1] != '.':
        ensemble_energy_3_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_overhang(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_3_overhang_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_3_overhang_guide = pfunc(strands=[max_3_overhang_seq_guide, RNA_reverse_complement(max_3_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_3_overhang_guide = partition_function_3_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[len_scaffold+len_spacer-1] != '.':
        ensemble_energy_3_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_overhang(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_3_overhang_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_3_overhang_seq_target = RNA_to_DNA(max_3_overhang_seq_target)
        partition_function_3_overhang_target = pfunc(strands=[DNA_3_overhang_seq_target, DNA_reverse_complement(DNA_3_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_3_overhang_target = partition_function_3_overhang_target[1]

    if ensemble_energy_3_overhang_guide > max_guide_energy_3_overhang:
        max_guide_energy_3_overhang = ensemble_energy_3_overhang_guide
    if ensemble_energy_3_overhang_guide < min_guide_energy_3_overhang:
        min_guide_energy_3_overhang = ensemble_energy_3_overhang_guide

    if ensemble_energy_3_overhang_target > max_target_energy_3_overhang:
        max_target_energy_3_overhang = ensemble_energy_3_overhang_target
    if ensemble_energy_3_overhang_target < min_target_energy_3_overhang:
        min_target_energy_3_overhang = ensemble_energy_3_overhang_target
            

    # Calculate/compare ensemble_energy_5_paired_guide and ensemble_energy_5_paired_target
    if str(subopt_structures_guide[0].structure)[len_scaffold] == '.':
        ensemble_energy_5_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_paired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_5_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_5_paired_guide = pfunc(strands=[max_5_paired_seq_guide, RNA_reverse_complement(max_5_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_5_paired_guide = partition_function_5_paired_guide[1]

    if str(subopt_structures_target[0].structure)[len_scaffold] == '.':
        ensemble_energy_5_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_paired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_5_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_5_paired_seq_target = RNA_to_DNA(max_5_paired_seq_target)
        partition_function_5_paired_target = pfunc(strands=[DNA_5_paired_seq_target, DNA_reverse_complement(DNA_5_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_5_paired_target = partition_function_5_paired_target[1]

    if ensemble_energy_5_paired_guide > max_guide_energy_5_paired:
        max_guide_energy_5_paired = ensemble_energy_5_paired_guide
    if ensemble_energy_5_paired_guide < min_guide_energy_5_paired:
        min_guide_energy_5_paired = ensemble_energy_5_paired_guide

    if ensemble_energy_5_paired_target > max_target_energy_5_paired:
        max_target_energy_5_paired = ensemble_energy_5_paired_target
    if ensemble_energy_5_paired_target < min_target_energy_5_paired:
        min_target_energy_5_paired = ensemble_energy_5_paired_target

    # Calculate/compare ensemble_energy_3_paired_guide and ensemble_energy_3_paired_target
    if str(subopt_structures_guide[0].structure)[len_scaffold+len_spacer-1] == '.':
        ensemble_energy_3_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_paired(str(subopt_structures_guide[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_3_paired_seq_guide = guide[max_start_index_guide+len_scaffold:max_start_index_guide+max_length_guide+len_scaffold]
        partition_function_3_paired_guide = pfunc(strands=[max_3_paired_seq_guide, RNA_reverse_complement(max_3_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_3_paired_guide = partition_function_3_paired_guide[1]

    if str(subopt_structures_target[0].structure)[len_scaffold+len_spacer-1] == '.':
        ensemble_energy_3_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_paired(str(subopt_structures_target[0].structure)[len_scaffold:len_scaffold+len_spacer])
        max_3_paired_seq_target = guide[max_start_index_target+len_scaffold:max_start_index_target+max_length_target+len_scaffold]
        DNA_3_paired_seq_target = RNA_to_DNA(max_3_paired_seq_target)
        partition_function_3_paired_target = pfunc(strands=[DNA_3_paired_seq_target, DNA_reverse_complement(DNA_3_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_3_paired_target = partition_function_3_paired_target[1]

    if ensemble_energy_3_paired_guide > max_guide_energy_3_paired:
        max_guide_energy_3_paired = ensemble_energy_3_paired_guide
    if ensemble_energy_3_paired_guide < min_guide_energy_3_paired:
        min_guide_energy_3_paired = ensemble_energy_3_paired_guide

    if ensemble_energy_3_paired_target > max_target_energy_3_paired:
        max_target_energy_3_paired = ensemble_energy_3_paired_target
    if ensemble_energy_3_paired_target < min_target_energy_3_paired:
        min_target_energy_3_paired = ensemble_energy_3_paired_target

    # Calculate/compare target seed region free energy 
    seed_region = target_truncated[14:20]
    subopt_structures_seed = subopt(strands=[seed_region, DNA_reverse_complement(seed_region)], energy_gap=6, model=my_model_DNA)
    seed_energy = subopt_structures_seed[0].energy

    if seed_energy > max_seed_energy:
        max_seed_energy = seed_energy
    if seed_energy < min_seed_energy:
        min_seed_energy = seed_energy

    # Calculate/compare target middle region free energy 
    middle_region = target_truncated[7:14]
    subopt_structures_middle = subopt(strands=[middle_region, DNA_reverse_complement(middle_region)], energy_gap=6, model=my_model_DNA)
    middle_energy = subopt_structures_middle[0].energy

    if middle_energy > max_middle_energy:
        max_middle_energy = middle_energy
    if middle_energy < min_middle_energy:
        min_middle_energy = middle_energy

    # Calculate/compare target distal region free energy 
    distal_region = target_truncated[0:7]
    subopt_structures_distal = subopt(strands=[distal_region, DNA_reverse_complement(distal_region)], energy_gap=6, model=my_model_DNA)
    distal_energy = subopt_structures_distal[0].energy

    if distal_energy > max_distal_energy:
        max_distal_energy = distal_energy
    if distal_energy < min_distal_energy:
        min_distal_energy = distal_energy

    to_be_added = [normalize_guide(subopt_structures_guide[0].energy + 3.57)]
    to_be_added.append(normalize_guide_conse(ensemble_energy_max_paired_guide))
    to_be_added.append(normalize_guide_conse_unpaired(ensemble_energy_max_unpaired_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_5_overhang_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_3_overhang_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_5_paired_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_3_paired_guide))
    
    to_be_added.append(normalize_target(subopt_structures_target[0].energy))
    to_be_added.append(normalize_target_conse(ensemble_energy_max_paired_target))
    to_be_added.append(normalize_target_conse_unpaired(ensemble_energy_max_unpaired_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_5_overhang_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_3_overhang_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_5_paired_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_3_paired_target))
    
    to_be_added.append(normalize_seed(seed_energy))
    to_be_added.append(normalize_middle(middle_energy))
    to_be_added.append(normalize_distal(distal_energy))

    # ==========================================================
    # Add Gaussian noise independently to each feature
    # ==========================================================
    
    for feature_idx in range(num_features):
    
        if i in feature_noise_indices[feature_idx]:
    
            noise = np.random.normal(
                loc=0.0,
                scale=NOISE_STD
            )
    
            to_be_added[feature_idx] += noise
    
            # Keep normalized feature in [0,1]
            to_be_added[feature_idx] = np.clip(
                to_be_added[feature_idx],
                0.0,
                1.0
            )

    energy_array_unit.append(to_be_added)

    energy_array_to_append = [energy_array_unit]
    energy_array.append(energy_array_to_append)

energy_array = list(itertools.chain.from_iterable(energy_array))

# Open a file in write mode
with open('Feature_MLP_0025.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in energy_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')
        

print('-------------')

print(min_guide_energy, max_guide_energy, min_target_energy, max_target_energy)
print(min_guide_energy_conse, max_guide_energy_conse, min_target_energy_conse, max_target_energy_conse)
print(min_guide_energy_conse_unpaired, max_guide_energy_conse_unpaired, min_target_energy_conse_unpaired, max_target_energy_conse_unpaired)
print(min_guide_energy_5_overhang, max_guide_energy_5_overhang, min_target_energy_5_overhang, max_target_energy_5_overhang)
print(min_guide_energy_3_overhang, max_guide_energy_3_overhang, min_target_energy_3_overhang, max_target_energy_3_overhang)
print(min_guide_energy_5_paired, max_guide_energy_5_paired, min_target_energy_5_paired, max_target_energy_5_paired)
print(min_guide_energy_3_paired, max_guide_energy_3_paired, min_target_energy_3_paired, max_target_energy_3_paired)

print(max_seed_energy, min_seed_energy)
print(max_middle_energy, min_middle_energy)
print(max_distal_energy, min_distal_energy)